# Weather in Szeged - Comprehensive Data Analysis Report

This notebook contains a complete analysis of the Weather in Szeged dataset including:
- Dataset statistics
- Feature visualization
- Correlation analysis
- Linear regression with confidence/prediction intervals
- ANOVA analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, f_oneway
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from IPython.display import display, HTML, Image
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## Часть 1: Статистика по датасету

In [ ]:
# Load data
df = pd.read_csv('/workspace/data/weatherHistory.csv')

# Statistics
n_objects = len(df)
n_features = len(df.columns)
precip_type_counts = df['Precip Type'].value_counts(dropna=False)
nan_percentage = (df.isnull().any(axis=1).sum() / n_objects) * 100

# Display statistics table
stats_data = {
    'Показатель': [
        'Число объектов в датасете',
        'Число признаков',
        'Число целевых классов по Precip Type',
        'Процент объектов с NaN значениями'
    ],
    'Значение': [
        n_objects,
        n_features,
        df['Precip Type'].nunique(dropna=True),
        f'{nan_percentage:.2f}%'
    ]
}
stats_df = pd.DataFrame(stats_data)
display(HTML('<h4>Общая статистика датасета</h4>'))
display(stats_df.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

In [ ]:
# Precip Type classes
display(HTML('<h4>Число объектов в каждом классе Precip Type</h4>'))
precip_table = pd.DataFrame({
    'Класс': precip_type_counts.index.tolist(),
    'Количество объектов': precip_type_counts.values
})
display(precip_table.style.hide(axis='index'))

In [ ]:
# NaN statistics
display(HTML('<h4>NaN значения по столбцам</h4>'))
nan_stats = df.isnull().sum()
nan_stats_df = pd.DataFrame({
    'Столбец': nan_stats.index,
    'NaN значений': nan_stats.values,
    'Процент NaN': (nan_stats.values / n_objects * 100).round(2)
})
display(nan_stats_df[nan_stats_df['NaN значений'] > 0].style.hide(axis='index'))

## Часть 2: Выбор признаков и очистка данных

In [ ]:
# Select features
selected_features = ['Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Pressure (millibars)']
target = 'Precip Type'

print(f"Выбранные количественные признаки: {selected_features}")
print(f"Целевой класс: {target}")

# Create and clean dataset
df_selected = df[selected_features + [target]].copy()
df_clean = df_selected.dropna()

print(f"\nЧисло объектов до очистки: {len(df_selected)}")
print(f"Число объектов после удаления NaN: {len(df_clean)}")
print(f"Удалено объектов: {len(df_selected) - len(df_clean)}")

# Classes
display(HTML('<h4>Классы в очищенном датасете</h4>'))
display(df_clean[target].value_counts().to_frame())

## Часть 3: Визуализация датасета (Pairplot)

In [ ]:
display(Image(filename='/workspace/output/pairplot_detailed.png'))

## Часть 4: Корреляционный анализ

In [ ]:
def calculate_correlations(data, features):
    n = len(features)
    pearson_corr = np.zeros((n, n))
    pearson_pval = np.zeros((n, n))
    spearman_corr = np.zeros((n, n))
    spearman_pval = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i == j:
                pearson_corr[i, j] = 1.0
                spearman_corr[i, j] = 1.0
            else:
                x = data[features[i]].values
                y = data[features[j]].values
                pc, pp = pearsonr(x, y)
                sc, sp = spearmanr(x, y)
                pearson_corr[i, j] = pc
                pearson_pval[i, j] = pp
                spearman_corr[i, j] = sc
                spearman_pval[i, j] = sp
    
    return pearson_corr, pearson_pval, spearman_corr, spearman_pval

# Entire dataset
display(HTML('<h3>Корреляции на всем датасете</h3>'))
pc, pp, sc, sp = calculate_correlations(df_clean, selected_features)

# Pearson table
pearson_df = pd.DataFrame(pc, index=[f[:15] for f in selected_features], 
                          columns=[f[:15] for f in selected_features])
display(HTML('<h4>Корреляция Пирсона</h4>'))
display(pearson_df.style.format('{:.3f}').background_gradient(cmap='coolwarm', vmin=-1, vmax=1))

# Spearman table
spearman_df = pd.DataFrame(sc, index=[f[:15] for f in selected_features], 
                           columns=[f[:15] for f in selected_features])
display(HTML('<h4>Корреляция Спирмана</h4>'))
display(spearman_df.style.format('{:.3f}').background_gradient(cmap='coolwarm', vmin=-1, vmax=1))

In [ ]:
display(HTML('<h4>Тепловые карты корреляций (весь датасет)</h4>'))
display(Image(filename='/workspace/output/correlation_heatmaps_all.png'))

In [ ]:
display(HTML('<h4>Тепловые карты корреляций (класс rain)</h4>'))
display(Image(filename='/workspace/output/correlation_heatmaps_rain.png'))

In [ ]:
display(HTML('<h4>Тепловые карты корреляций (класс snow)</h4>'))
display(Image(filename='/workspace/output/correlation_heatmaps_snow.png'))

## Часть 5: Линейная регрессия с прогнозными интервалами (95% Prediction Intervals)

In [ ]:
display(Image(filename='/workspace/output/regression_prediction_intervals.png'))

## Часть 6: Линейная регрессия с доверительными интервалами (95% Confidence Intervals)

In [ ]:
display(Image(filename='/workspace/output/regression_confidence_intervals.png'))

## Часть 7: Формулы регрессий и статистические критерии

In [ ]:
# Load regression summary
regression_df = pd.read_csv('/workspace/output/regression_summary.csv')
display(HTML('<h4>Сводная таблица регрессий</h4>'))
display(regression_df)

In [ ]:
# Detailed regression output
classes = df_clean[target].unique()

for precip_type in classes:
    display(HTML(f'<h3>Класс: {precip_type.upper()}</h3>'))
    df_class = df_clean[df_clean[target] == precip_type]
    
    results = []
    for i, feat_i in enumerate(selected_features):
        for j, feat_j in enumerate(selected_features):
            if i != j:
                x = df_class[feat_j].values
                y = df_class[feat_i].values
                
                X = sm.add_constant(x)
                model = sm.OLS(y, X).fit()
                
                # LR test
                ll_full = model.llf
                n = model.nobs
                y_mean = np.mean(y)
                residuals_null = y - y_mean
                sigma2_null = np.sum(residuals_null**2) / n
                ll_null = -n/2 * (np.log(2*np.pi*sigma2_null) + 1)
                lr_stat = 2 * (ll_full - ll_null)
                lr_pvalue = 1 - stats.chi2.cdf(lr_stat, 1)
                
                results.append({
                    'Y': feat_i[:15],
                    'X': feat_j[:15],
                    'Формула': f'{model.params[0]:.4f} + ({model.params[1]:.6f}) * X',
                    'R²': f'{model.rsquared:.4f}',
                    'F-stat (Wald)': f'{model.fvalue:.2f}',
                    'F p-value': f'{model.f_pvalue:.2e}',
                    'LR χ²': f'{lr_stat:.2f}',
                    'LR p-value': f'{lr_pvalue:.2e}'
                })
    
    display(pd.DataFrame(results))

## Часть 8: Дисперсионный анализ (ANOVA)

In [ ]:
anova_feature = 'Temperature (C)'
display(HTML(f'<h4>Выбранный признак для ANOVA: {anova_feature}</h4>'))

# One-way ANOVA
groups = [df_clean[df_clean[target] == c][anova_feature].values for c in classes]
f_stat, p_value = f_oneway(*groups)

anova_results = pd.DataFrame({
    'Показатель': ['F-статистика', 'df_between', 'df_within', 'p-значение'],
    'Значение': [f'{f_stat:.4f}', len(classes) - 1, len(df_clean) - len(classes), f'{p_value:.2e}']
})
display(HTML('<h4>One-way ANOVA для всех классов</h4>'))
display(anova_results.style.hide(axis='index'))

In [ ]:
# Pairwise comparisons
from itertools import combinations

pairwise_results = []
class_list = list(classes)
for c1, c2 in combinations(class_list, 2):
    group1 = df_clean[df_clean[target] == c1][anova_feature].values
    group2 = df_clean[df_clean[target] == c2][anova_feature].values
    f_pair, p_pair = f_oneway(group1, group2)
    pairwise_results.append({
        'Сравнение': f'{c1} vs {c2}',
        'F-статистика': f'{f_pair:.4f}',
        'df_between': 1,
        'df_within': len(group1) + len(group2) - 2,
        'p-значение': f'{p_pair:.2e}'
    })

display(HTML('<h4>Попарные сравнения классов</h4>'))
display(pd.DataFrame(pairwise_results))

In [ ]:
# Display class statistics
display(HTML('<h4>Статистика по классам</h4>'))
class_stats = []
for precip_type in classes:
    data = df_clean[df_clean[target] == precip_type][anova_feature].values
    median = np.median(data)
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    mean = np.mean(data)
    se = stats.sem(data)
    ci_low, ci_high = stats.t.interval(0.95, len(data)-1, loc=mean, scale=se)
    
    class_stats.append({
        'Класс': precip_type,
        'N': len(data),
        'Медиана': f'{median:.2f}',
        'IQR': f'{iqr:.2f}',
        '95% CI (нижняя)': f'{ci_low:.2f}',
        '95% CI (верхняя)': f'{ci_high:.2f}'
    })

display(pd.DataFrame(class_stats))

In [ ]:
display(HTML('<h4>Raincloud Plot</h4>'))
display(Image(filename='/workspace/output/anova_raincloud.png'))

In [ ]:
display(HTML('<h4>Raincloud Plot (ptitprince)</h4>'))
display(Image(filename='/workspace/output/anova_raincloud_ptitprince.png'))

---
## Заключение

Анализ датасета Weather in Szeged выполнен полностью. Основные результаты:

1. **Датасет содержит 96,453 наблюдений и 12 признаков** с двумя классами осадков (rain и snow)
2. **Выбраны 4 количественных признака**: Temperature, Humidity, Wind Speed, Pressure
3. **Корреляционный анализ** показал сильную отрицательную корреляцию между Temperature и Humidity (r = -0.63 для rain)
4. **Линейные регрессии** построены для всех пар признаков с прогнозными и доверительными интервалами
5. **ANOVA анализ** подтвердил статистически значимые различия в температуре между классами rain и snow (F = 44636, p < 0.001)